In [1]:
# -*- coding: utf-8 -*-
import os, sys, re, json, math, time, random, datetime
from pathlib import Path
from typing import Dict, Any, Optional

# ---------------- Pricing (USD per 1M tokens) ----------------
# 최신 가격은 OpenAI 공식 문서를 참고하세요.
PRICING = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

# ---------------- OpenAI SDK ----------------
try:
    from openai import OpenAI
except ImportError:
    print("`pip install openai`를 먼저 설치해주세요.")
    raise

# ---------------- Token estimator ----------------
def approx_tokens(text: str) -> int:
    """간단한 텍스트 토큰 수 추정 함수"""
    try:
        import tiktoken
        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(text or ""))
    except ImportError:
        if not text: return 0
        hangul = len(re.findall(r"[가-힣]", text))
        # 한글 비율에 따라 토큰당 글자 수 어림 계산 (경험적 수치)
        ratio = hangul / len(text) if len(text) > 0 else 0
        divisor = 2.6 if ratio >= 0.3 else 3.5
        return math.ceil(len(text) / divisor)

# ---------------- Cost helpers ----------------
def calc_cost_usd(model: str, in_tokens: int, out_tokens: int) -> float:
    """토큰 사용량 기반 비용 계산"""
    if model not in PRICING:
        raise ValueError(f"'{model}' 모델의 가격 정보가 없습니다. PRICING 딕셔너리에 추가해주세요.")
    p = PRICING[model]
    return (in_tokens / 1_000_000) * p["input"] + (out_tokens / 1_000_000) * p["output"]

def estimate_pre_cost(jsonl_path: str, model: str, per_product_out_tokens: int = 80) -> Dict[str, Any]:
    """실행 전 전체 비용 사전 추정"""
    total_in = 0
    total_out = 0
    n_lines = 0
    n_products = None
    prod_pat = re.compile(r"^\s*-\s*\d+\.\s", re.MULTILINE)

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            prompt = rec.get("prompt", "")
            total_in += approx_tokens(prompt)
            if n_products is None:
                m = prod_pat.findall(prompt)
                n_products = len(m) if m else 10
            n_lines += 1

    if n_products is None: n_products = 10
    total_out = n_lines * n_products * per_product_out_tokens
    cost = calc_cost_usd(model, total_in, total_out)

    return {
        "lines": n_lines,
        "products_per_call_est": n_products,
        "total_input_tokens_est": total_in,
        "total_output_tokens_est": total_out,
        "total_cost_usd_est": round(cost, 4),
        "model": model,
        "per_product_out_tokens": per_product_out_tokens,
    }

# ---------------- JSON safe parse ----------------
def safe_parse_json(s: str) -> Optional[Dict[str, Any]]:
    """LLM이 반환한 텍스트에서 JSON 부분만 안전하게 파싱"""
    s = (s or "").strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    # 마크다운 코드 블록(` ```json ... ``` `) 안에 있는 경우 대응
    if s.startswith("```") and s.endswith("```"):
        s = s.split('\n', 1)[1].rsplit('\n', 1)[0]
    
    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(s[start:end+1])
        except Exception:
            return None
    return None

# ---------------- LLM call (with retries) ----------------
def call_llm(client: OpenAI, model: str, prompt: str, max_output_tokens: int = 800, temperature: float = 0.2):
    """OpenAI API 호출"""
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_output_tokens,
        response_format={"type": "json_object"}, # JSON 모드 활성화
        messages=[
            {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
            {"role": "user", "content": prompt},
        ],
    )
    text = resp.choices[0].message.content or ""
    usage = resp.usage
    in_tok = getattr(usage, "prompt_tokens", 0) or 0
    out_tok = getattr(usage, "completion_tokens", 0) or 0
    return text, in_tok, out_tok

def call_with_retries(client, model, prompt, max_output_tokens, temperature, max_retries=5, base_delay=1.0):
    """네트워크 에러 발생 시 재시도 로직 포함"""
    for i in range(max_retries):
        try:
            return call_llm(client, model, prompt, max_output_tokens, temperature)
        except Exception as e:
            retriable = any(x in str(e).lower() for x in ["rate", "timeout", "502", "503", "504", "temporarily"])
            if not retriable or i == max_retries - 1:
                print(f"API 호출 실패 (재시도 불가): {e}")
                raise
            print(f"API 호출 오류 (재시도 {i+1}/{max_retries}): {e}")
            time.sleep(base_delay * (2 ** i) * (1.0 + 0.25 * random.random()))

# ---------------- Runner ----------------
def run(
    input_jsonl: str,
    output_jsonl: Optional[str],
    model: str,
    api_key: Optional[str],
    budget_usd: float,
    per_product_out_tokens: int,
    max_output_tokens: int,
    temperature: float,
    dry_run: bool,
):
    """메인 실행 함수"""
    if api_key:
        client = OpenAI(api_key=api_key)
    else:
        # 환경변수에서 API 키를 읽어옴
        client = OpenAI()

    in_path = Path(input_jsonl)
    if not in_path.exists():
        # sys.exit 대신 에러를 발생시켜 노트북 실행을 중단
        raise FileNotFoundError(f"[ERR] 입력 JSONL 파일을 찾을 수 없습니다: {in_path}")

    # 0) 사전 비용 추정
    pre = estimate_pre_cost(str(in_path), model, per_product_out_tokens=per_product_out_tokens)
    print("=== 사전 비용 추정 ===")
    for k, v in pre.items():
        print(f"- {k}: {v}")
    print("-" * 20)
    
    if pre['total_cost_usd_est'] > budget_usd:
        print(f"경고: 예상 비용(${pre['total_cost_usd_est']:.4f})이 예산(${budget_usd:.4f})을 초과합니다.")
        # 사용자 확인을 위해 input()을 사용할 수 있으나, 자동 실행을 위해 일단 진행
        # if input("계속 진행하시겠습니까? (y/n): ").lower() != 'y':
        #     print("작업을 중단합니다.")
        #     return

    # 1) 출력 경로 설정
    if not output_jsonl:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_jsonl = f"results_monthly_generic_{stamp}.jsonl"
    out_path = Path(output_jsonl)

    total_in = 0
    total_out = 0
    total_cost = 0.0
    processed = 0

    with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", encoding="utf-8") as fout:
        # 전체 라인 수를 알기 위해 파일을 한번 읽음 (진행률 표시용)
        num_lines = sum(1 for line in fin if line.strip())
        fin.seek(0) # 파일 포인터를 다시 처음으로
        
        print(f"총 {num_lines}개의 프롬프트 처리 시작...")
        
        for i, line in enumerate(fin):
            if not line.strip(): continue
            rec = json.loads(line)
            persona_id = rec.get("persona_id")
            prompt = rec.get("prompt", "")
            if not prompt:
                continue

            # 진행률 출력
            print(f"[{i+1}/{num_lines}] Persona: {persona_id} | 누적 비용: ${total_cost:.4f}", end='\r')

            # 선제 예산 체크
            est_in = approx_tokens(prompt)
            est_pre_cost = calc_cost_usd(model, est_in, 0)
            if total_cost + est_pre_cost > budget_usd:
                print(f"\n[중단] 예산(${budget_usd}) 초과가 예상되어 작업을 중단합니다. (현재 누적 비용: ${total_cost:.4f})")
                break

            if dry_run:
                # dry_run 모드: 실제 API 호출 없이 출력 파일 구조만 생성
                out_obj = { "persona_id": persona_id, "note": "DRY_RUN" }
                fout.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
                processed += 1
                continue

            # 실제 LLM 호출
            try:
                raw, in_tok, out_tok = call_with_retries(
                    client, model, prompt, max_output_tokens=max_output_tokens, temperature=temperature
                )
            except Exception as e:
                out_obj = { "persona_id": persona_id, "error": str(e) }
                fout.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
                continue

            cost = calc_cost_usd(model, in_tok, out_tok)
            total_in += in_tok
            total_out += out_tok
            total_cost += cost

            parsed = safe_parse_json(raw)
            out_obj = {
                "persona_id": persona_id,
                "raw_text": raw,
                "parsed_json": parsed,
                "usage": {"prompt_tokens": in_tok, "completion_tokens": out_tok, "total_tokens": in_tok + out_tok},
                "cost_usd": round(cost, 6),
            }
            fout.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
            processed += 1

            if total_cost >= budget_usd:
                print(f"\n[중단] 예산 한도(${budget_usd})에 도달하여 작업을 중단합니다.")
                break

    print("\n" + "="*20 + " 실행 요약 " + "="*20)
    print(f"- 결과 파일: {out_path}")
    print(f"- 처리된 프롬프트 수: {processed} / {num_lines}")
    print(f"- 총 입력 토큰: {total_in}")
    print(f"- 총 출력 토큰: {total_out}")
    print(f"- 최종 누적 비용 (USD): ${total_cost:.4f}")


In [2]:
################################################################################
# --- 주피터 노트북 실행부 ---
# 아래 설정값을 자신의 환경에 맞게 수정한 후, 이 셀을 실행하세요.
################################################################################

# ##### 설정값 #####
# 1. 입력 파일 경로 (이전에 생성한 프롬프트 jsonl 파일)
INPUT_FILE = "prompts_monthly_generic_20250821_150109.jsonl"  # <-- ⚠️ 여기에 파일 경로를 정확히 입력하세요.

# 2. OpenAI API 키
# "sk-..." 형태의 키를 직접 입력하거나, 환경 변수에 등록했다면 None으로 두세요.
import os
import dotenv
dotenv.load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")  # <-- ⚠️ 여기에 자신의 API 키를 입력하거나 None으로 두세요.

# 3. 최대 예산 (USD)
# 이 금액을 넘으면 작업이 자동으로 중단됩니다.
BUDGET_USD = 8.5

# 4. 사용할 모델
MODEL = "gpt-4o-mini"

# 5. (옵션) 결과 파일 경로
# None으로 두면 'results_...' 형태의 파일이 자동으로 생성됩니다.
OUTPUT_FILE = None 

# 6. (옵션) 기타 설정값
# 사전 비용 추정 시, 제품 1개당 예상되는 출력 토큰 수
PER_PRODUCT_OUT_TOKENS_EST = 80 
# 실제 API 호출 시, 최대 출력 토큰 제한
MAX_OUTPUT_TOKENS = 1200
# 모델의 창의성/일관성 조절 (낮을수록 일관적)
TEMPERATURE = 0.1
# True로 설정하면 실제 API를 호출하지 않고 실행 과정만 점검합니다.
DRY_RUN = False 
# ##### 설정 끝 #####


# --- 실행 ---
try:
    # 설정값을 run 함수에 전달하여 실행
    run(
        input_jsonl=INPUT_FILE,
        output_jsonl=OUTPUT_FILE,
        model=MODEL,
        api_key=API_KEY,
        budget_usd=BUDGET_USD,
        per_product_out_tokens=PER_PRODUCT_OUT_TOKENS_EST,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        temperature=TEMPERATURE,
        dry_run=DRY_RUN,
    )
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"예상치 못한 오류가 발생했습니다: {e}")

=== 사전 비용 추정 ===
- lines: 2156
- products_per_call_est: 15
- total_input_tokens_est: 3159211
- total_output_tokens_est: 2587200
- total_cost_usd_est: 2.0262
- model: gpt-4o-mini
- per_product_out_tokens: 80
--------------------
총 2156개의 프롬프트 처리 시작...
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.435
API 호출 실패 (재시도 불가): Connection error.912
API 호출 실패 (재시도 불가): Connection error.912
API 호출 실패 (재시도 불가): Connection error.912
API 호출 실패 (재시도 불가): Connection error.912
API 호출 실패 (재시도 불가): Connection error.912
API 호출 실패 (재시도 불가): Connection error.912
[2156/2156] Persona: 2156 | 누적 비용: $2.2246
==================== 실행 요약 ====================
- 